In [1]:
import re
from pathlib import Path
from collections import Counter
from pycocotools.coco import COCO

# -------- paths (as per your setup) --------
TRAIN_ANN = Path(r"C:\COCO\annotations\captions_train2014.json")
OUT_PATH  = Path(r"D:\PROJECT\labels.txt")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert TRAIN_ANN.exists(), f"Not found: {TRAIN_ANN}"

# -------- simple tokenizer --------
def tokenize(s: str):
    s = s.lower().strip()
    s = re.sub(r"[^a-z\s]", " ", s)  # keep letters + spaces only
    return [w for w in s.split() if w]

# -------- stopwords (lightweight built-in list) --------
STOPWORDS = set("""
a an the and or of in on at to for from with by is are was were be been being
this that these those it its as into over under up down out off if then than
not no yes very so too such just about across after again against all am among
any because before between both but can could did do does doing don during each
few had has have having he her hers him his how i im ive me more most my our ours
ourselves she should some their theirs them themselves there these they this those
through time under until we were what when where which while who whom why will
would you your yours yourself yourselves
""".split())

# Add common caption filler words
STOPWORDS |= set("""
photo picture image scene view showing shown sitting standing wearing holding
looking lookingat near next beside front behind left right top bottom
""".split())

# -------- build frequency list --------
coco = COCO(str(TRAIN_ANN))
counter = Counter()

for ann in coco.anns.values():
    words = tokenize(ann["caption"])
    for w in words:
        # filters
        if w in STOPWORDS:
            continue
        if len(w) < 3:
            continue
        # keep only alphabetic already, but double-check
        if not w.isalpha():
            continue
        counter[w] += 1

# choose top-N
TOP_N = 1500
labels = [w for w, f in counter.most_common(TOP_N)]

# write file
OUT_PATH.write_text("\n".join(labels) + "\n", encoding="utf-8")

print("Saved:", OUT_PATH)
print("Total labels:", len(labels))
print("Top 30 labels:", labels[:30])


loading annotations into memory...
Done (t=1.11s)
creating index...
index created!
Saved: D:\PROJECT\labels.txt
Total labels: 1500
Top 30 labels: ['man', 'two', 'people', 'white', 'woman', 'table', 'street', 'person', 'large', 'group', 'field', 'small', 'tennis', 'black', 'plate', 'room', 'train', 'dog', 'riding', 'red', 'young', 'cat', 'water', 'baseball', 'walking', 'playing', 'bathroom', 'sign', 'blue', 'food']


In [2]:
import torch, re
from pathlib import Path
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")
CKPT_PATH = RUN_DIR / "checkpoint_last.pt"
VOCAB_PATH = RUN_DIR / "vocab.pt"

# Load vocab
v = torch.load(VOCAB_PATH, map_location="cpu")
stoi, itos = v["stoi"], v["itos"]

SPECIALS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<DIV0>", "<DIV1>", "<DIV2>"]
PAD, BOS, EOS, UNK, DIV0, DIV1, DIV2 = SPECIALS

# same transforms
tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ---- model definition must match training notebook ----
class ResNetEncoder(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        base = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.proj = nn.Conv2d(2048, d_model, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.proj(feat)
        return feat.flatten(2).permute(2,0,1)

class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.1, max_len=64):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                                           dropout=dropout, batch_first=False)
        self.dec = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt_ids, memory, tgt_key_padding_mask=None):
        B,T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0).expand(B,T)
        x = self.tok_emb(tgt_ids) + self.pos_emb(pos)
        x = x.permute(1,0,2)
        causal = torch.triu(torch.ones(T,T, device=tgt_ids.device), diagonal=1).bool()
        h = self.dec(tgt=x, memory=memory, tgt_mask=causal, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.out(h).permute(1,0,2)

class CaptionModel(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.enc = ResNetEncoder(d_model=d_model)
        self.dec = CaptionDecoder(vocab_size=vocab_size, d_model=d_model)

    def forward(self, images, cap_inp, cap_pad_mask=None):
        mem = self.enc(images)
        return self.dec(cap_inp, mem, tgt_key_padding_mask=cap_pad_mask)

model = CaptionModel(vocab_size=len(itos), d_model=512).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

print("Loaded:", CKPT_PATH, "| epoch:", ckpt.get("epoch"))


DEVICE: cuda
Loaded: D:\PROJECT\runs\c3dc_baseline_20260210_084907\checkpoint_last.pt | epoch: 10


In [3]:
!pip install open_clip_torch


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 1.7 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.5 MB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.6 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.6 MB 1.9 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.6 MB 1.6 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.6 MB 1.6 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.6 MB 1.6 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/


[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install open_clip_torch


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
print("PYTHON:", sys.executable)


PYTHON: C:\Users\Student\.conda\envs\icap_gpu\python.exe


In [8]:
import sys
!{sys.executable} -m pip install -U open_clip_torch


  Using cached open_clip_torch-3.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
Using cached open_clip_torch-3.2.0-py3-none-any.whl (1.5 MB)
Using cached timm-1.0.24-py3-none-any.whl (2.6 MB)

   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   -------------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Student\.conda\envs\icap_gpu\python.exe -m pip install --upgrade pip


In [9]:
import open_clip


C:\Users\Student\.conda\envs\icap_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
import open_clip
import torch

LABELS_TXT = Path(r"D:\PROJECT\labels.txt")
assert LABELS_TXT.exists(), f"labels.txt not found: {LABELS_TXT}"

class ClipPseudoContext:
    def __init__(self, labels_txt, clip_model="ViT-B-32", pretrained="openai"):
        self.labels = [l.strip() for l in open(labels_txt, "r", encoding="utf-8") if l.strip()]
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(clip_model, pretrained=pretrained)
        self.tokenizer = open_clip.get_tokenizer(clip_model)

        self.model = self.model.to(DEVICE).eval()
        for p in self.model.parameters():
            p.requires_grad = False

        # precompute text features once
        #prompts = [f"a photo of {w}" for w in self.labels]  # keep prompt simple
        self.labels = self.labels[:300]   # quick debug (later remove)
        prompts = [f"a photo of {w}" for w in self.labels]
        with torch.no_grad():
            text = self.tokenizer(prompts).to(DEVICE)
            text_feat = self.model.encode_text(text)
            self.text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def get_context(self, pil_image, topN=25):
        img = self.preprocess(pil_image).unsqueeze(0).to(DEVICE)
        img_feat = self.model.encode_image(img)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        sims = (img_feat @ self.text_feat.T).squeeze(0)  # (L,)
        vals, idx = torch.topk(sims, k=min(topN, sims.numel()))
        chosen = [self.labels[i] for i in idx.tolist()]

        # confidence margin
        s1 = vals[0].item()
        s5 = vals[min(4, len(vals)-1)].item()
        conf = s1 - s5
        return set(chosen), conf


In [11]:
import math

def sigmoid(x): 
    return 1/(1+math.exp(-x))

def conf_to_lambda(conf, tau=0.08, k=25.0):
    return sigmoid(k * (conf - tau))

# Precompute ids of "content-like" tokens (alphabetic, not specials)
CONTENT_IDS = []
for vid, tok in enumerate(itos):
    if tok in SPECIALS:
        continue
    if tok.isalpha():
        CONTENT_IDS.append(vid)
CONTENT_IDS = torch.tensor(CONTENT_IDS, dtype=torch.long, device=DEVICE)

print("content ids:", len(CONTENT_IDS))


content ids: 8790


In [12]:
@torch.no_grad()
def sample_decode_c3dc(model, image_tensor, stoi, itos, div_token,
                       allowed_set=None, conf=None,
                       topN=25, alpha=0.35, tau=0.08, k=25.0,
                       max_len=30, temperature=1.0, top_k=0, top_p=1.0,
                       rep_penalty=1.0, block_bigrams=True):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    pad_id = stoi[PAD]
    bos_id = stoi[BOS]
    eos_id = stoi[EOS]
    div_id = stoi[div_token]

    lam = 0.0
    if (allowed_set is not None) and (conf is not None):
        lam = conf_to_lambda(conf, tau=tau, k=k)

    seq = [div_id, bos_id]
    used_bigrams = set()

    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(pad_id))[:, -1, :].squeeze(0)

        # repetition penalty
        if rep_penalty and rep_penalty > 1.0:
            for prev in set(seq):
                logits[prev] /= rep_penalty

        # bigram block
        if block_bigrams and len(seq) >= 2:
            prev_tok = seq[-1]
            # only block for CONTENT_IDS to keep it light
            for cand in CONTENT_IDS.tolist():
                if (prev_tok, cand) in used_bigrams:
                    logits[cand] = -1e9

        # calibrated pseudo-context penalty (soft)
        if allowed_set is not None and conf is not None:
            # penalize content tokens that are NOT in allowed_set
            # (vectorized mask over CONTENT_IDS)
            toks = [itos[i] for i in CONTENT_IDS.tolist()]
            mask = torch.tensor([t not in allowed_set for t in toks], device=DEVICE)
            logits[CONTENT_IDS[mask]] -= (lam * alpha)

        logits = logits / max(1e-6, temperature)
        probs = F.softmax(logits, dim=-1)

        # top-k
        if top_k and top_k > 0:
            v, idx = torch.topk(probs, top_k)
            probs2 = torch.zeros_like(probs)
            probs2[idx] = v
            probs = probs2 / probs2.sum()

        # nucleus top-p
        if top_p < 1.0:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            csum = torch.cumsum(sorted_probs, dim=0)
            cut = (csum > top_p).nonzero(as_tuple=False)
            if cut.numel() > 0:
                last = cut[0].item()
                keep = sorted_idx[: last+1]
                probs2 = torch.zeros_like(probs)
                probs2[keep] = probs[keep]
                probs = probs2 / probs2.sum()

        next_id = int(torch.multinomial(probs, 1).item())
        seq.append(next_id)
        used_bigrams.add((seq[-2], seq[-1]))

        if next_id == eos_id:
            break

    # decode
    words = []
    for i in seq:
        tok = itos[i]
        if tok == EOS: break
        if tok in SPECIALS: continue
        words.append(tok)
    return " ".join(words), float(lam)


In [15]:
import torch, open_clip
from pathlib import Path

class ClipPseudoContext:
    def __init__(self, labels_txt, clip_model="ViT-B-32", pretrained="openai", cache_dir=Path(r"D:\PROJECT\clip_cache")):
        cache_dir.mkdir(parents=True, exist_ok=True)

        self.labels = [l.strip() for l in open(labels_txt, "r", encoding="utf-8") if l.strip()]
        self.clip_model_name = clip_model
        self.pretrained = pretrained

        safe_name = f"{clip_model.replace('/','_')}_{pretrained}_L{len(self.labels)}"
        self.cache_path = cache_dir / f"textfeat_{safe_name}.pt"

        self.model, _, self.preprocess = open_clip.create_model_and_transforms(clip_model, pretrained=pretrained)
        self.tokenizer = open_clip.get_tokenizer(clip_model)
        self.model = self.model.to(DEVICE).eval()
        for p in self.model.parameters():
            p.requires_grad = False

        if self.cache_path.exists():
            obj = torch.load(self.cache_path, map_location=DEVICE)
            self.text_feat = obj["text_feat"]
            # labels stored so we know cache matches
            cached_labels = obj["labels"]
            assert cached_labels == self.labels, "labels.txt changed; delete cache file and rerun"
            print("Loaded cached text features:", self.cache_path)
        else:
            prompts = [f"a photo of {w}" for w in self.labels]
            with torch.no_grad():
                text = self.tokenizer(prompts).to(DEVICE)
                text_feat = self.model.encode_text(text)
                self.text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

            torch.save({"text_feat": self.text_feat.detach().cpu(), "labels": self.labels}, self.cache_path)
            # reload to DEVICE
            obj = torch.load(self.cache_path, map_location=DEVICE)
            self.text_feat = obj["text_feat"]
            print("Saved cached text features:", self.cache_path)

    @torch.no_grad()
    def get_context(self, pil_image, topN=25):
        img = self.preprocess(pil_image).unsqueeze(0).to(DEVICE)
        img_feat = self.model.encode_image(img)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        text_feat = self.text_feat.to(DEVICE)
        sims = (img_feat @ text_feat.T).squeeze(0)
        vals, idx = torch.topk(sims, k=min(topN, sims.numel()))
        chosen = [self.labels[i] for i in idx.tolist()]

        s1 = vals[0].item()
        s5 = vals[min(4, len(vals)-1)].item()
        conf = s1 - s5
        return set(chosen), conf


In [16]:
clip_ctx = ClipPseudoContext(LABELS_TXT)

# pick a val image -> PIL (for CLIP) and tensor (for caption model)
# We'll use the COCO val dataset metadata directly:
coco_val = COCO(str(VAL_ANN))
img_id = list(coco_val.imgs.keys())[0]
img_info = coco_val.loadImgs([img_id])[0]
pil = Image.open(VAL_IMG / img_info["file_name"]).convert("RGB")
img_t = tfm(pil)

allowed_set, conf = clip_ctx.get_context(pil, topN=25)
print("pseudo-context top10:", list(sorted(list(allowed_set)))[:10])
print("conf:", conf)

c0, lam0 = sample_decode_c3dc(model, img_t, stoi, itos, DIV0, allowed_set, conf,
                              temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.15)
c1, lam1 = sample_decode_c3dc(model, img_t, stoi, itos, DIV1, allowed_set, conf,
                              temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)
c2, lam2 = sample_decode_c3dc(model, img_t, stoi, itos, DIV2, allowed_set, conf,
                              temperature=1.1, top_k=0, top_p=0.9, rep_penalty=1.25)

print("\nλ (should be same):", lam0)
print("DIV0:", c0)
print("DIV1:", c1)
print("DIV2:", c2)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\Student\.conda\envs\icap_gpu\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Saved cached text features: D:\PROJECT\clip_cache\textfeat_ViT-B-32_openai_L1500.pt


NameError: name 'VAL_ANN' is not defined

In [ ]:
# quick cleanup: remove numeric/generic words
bad = {"two", "one", "three", "four", "five"}
labels = [l for l in open(LABELS_TXT, "r", encoding="utf-8").read().splitlines() if l not in bad]
Path(LABELS_TXT).write_text("\n".join(labels) + "\n", encoding="utf-8")
print("Cleaned labels.txt, now:", len(labels))


In [17]:
from pathlib import Path
from pycocotools.coco import COCO
from PIL import Image
from torchvision import transforms

# ---- your COCO paths ----
VAL_IMG = Path(r"C:\COCO\val2017\val2017")
VAL_ANN = Path(r"C:\COCO\annotations\captions_val2017.json")
assert VAL_IMG.exists(), VAL_IMG
assert VAL_ANN.exists(), VAL_ANN

# same transform you used for caption model input
tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# pick a val image (first id)
coco_val = COCO(str(VAL_ANN))
img_id = list(coco_val.imgs.keys())[0]
img_info = coco_val.loadImgs([img_id])[0]
pil = Image.open(VAL_IMG / img_info["file_name"]).convert("RGB")
img_t = tfm(pil)

allowed_set, conf = clip_ctx.get_context(pil, topN=25)
print("pseudo-context top10:", sorted(list(allowed_set))[:10])
print("conf:", conf)
print("lambda:", conf_to_lambda(conf, tau=0.08, k=25.0))

c0, lam0 = sample_decode_c3dc(model, img_t, stoi, itos, DIV0, allowed_set, conf,
                              temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.15)
c1, lam1 = sample_decode_c3dc(model, img_t, stoi, itos, DIV1, allowed_set, conf,
                              temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)
c2, lam2 = sample_decode_c3dc(model, img_t, stoi, itos, DIV2, allowed_set, conf,
                              temperature=1.1, top_k=0, top_p=0.9, rep_penalty=1.25)

print("\nλ (should be same):", lam0)
print("DIV0:", c0)
print("DIV1:", c1)
print("DIV2:", c2)


loading annotations into memory...
Done (t=0.19s)
creating index...
index created!
pseudo-context top10: ['apron', 'cabinet', 'cabinets', 'chef', 'chefs', 'cooked', 'cooking', 'costume', 'counter', 'counters']
conf: 0.014054298400878906
lambda: 0.1612924985854415

λ (should be same): 0.1612924985854415
DIV0: three men in a very large kitchen with lime boats
DIV1: three women doing the tasks on their stove
DIV2: three men working in a very large kitchen


In [18]:
allowed_set, conf = clip_ctx.get_context(pil, topN=25)
print("conf:", conf, "lambda:", conf_to_lambda(conf, tau=0.0, k=25.0))

c0, _ = sample_decode_c3dc(model, img_t, stoi, itos, DIV0, allowed_set, conf,
                           alpha=0.8, tau=0.0, k=25.0,
                           temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.15)
c1, _ = sample_decode_c3dc(model, img_t, stoi, itos, DIV1, allowed_set, conf,
                           alpha=0.8, tau=0.0, k=25.0,
                           temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)
c2, _ = sample_decode_c3dc(model, img_t, stoi, itos, DIV2, allowed_set, conf,
                           alpha=0.8, tau=0.0, k=25.0,
                           temperature=1.1, top_k=0, top_p=0.9, rep_penalty=1.25)

print("DIV0:", c0)
print("DIV1:", c1)
print("DIV2:", c2)


conf: 0.014054298400878906 lambda: 0.5869467206855068
DIV0: three women cooking in a large kitchen area
DIV1: three men in a kitchen preparing an oven
DIV2: three women preparing a meal in the kitchen with clear screen


In [19]:
# --- set tuned params here ---
ALPHA = 0.8
TAU   = 0.0
K     = 25.0
TOPN  = 25

metrics_200 = eval_c3dc_on_val(N=200, alpha=ALPHA, tau=TAU, k=K)
print(metrics_200)


NameError: name 'eval_c3dc_on_val' is not defined

In [20]:
import numpy as np
from collections import Counter
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image

def distinct_n(captions, n=2):
    total_ngrams = 0
    uniq_ngrams = set()
    for c in captions:
        toks = c.split()
        if len(toks) < n:
            continue
        for i in range(len(toks)-n+1):
            ng = tuple(toks[i:i+n])
            uniq_ngrams.add(ng)
            total_ngrams += 1
    return (len(uniq_ngrams) / max(1, total_ngrams))

def novel_content_rate(caption, allowed_set):
    toks = [t for t in caption.split() if t.isalpha()]
    if not toks:
        return 0.0
    bad = sum(1 for t in toks if t not in allowed_set)
    return bad / len(toks)

def eval_c3dc_on_val(N=200, alpha=0.8, tau=0.0, k=25.0, topN=25):
    coco_val = COCO(str(VAL_ANN))
    img_ids = list(coco_val.imgs.keys())[:N]

    all_caps = []
    ncrs = []

    for img_id in tqdm(img_ids, desc=f"c3dc eval {N}"):
        info = coco_val.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        allowed_set, conf = clip_ctx.get_context(pil, topN=topN)

        for div, cfg in [
            (DIV0, dict(temperature=1.0, top_k=0,  top_p=1.0, rep_penalty=1.15)),
            (DIV1, dict(temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)),
            (DIV2, dict(temperature=1.1, top_k=0,  top_p=0.9, rep_penalty=1.25)),
        ]:
            cap, _ = sample_decode_c3dc(
                model, img_t, stoi, itos, div,
                allowed_set, conf,
                alpha=alpha, tau=tau, k=k,
                **cfg
            )
            all_caps.append(cap)
            ncrs.append(novel_content_rate(cap, allowed_set))

    d1 = distinct_n(all_caps, n=1)
    d2 = distinct_n(all_caps, n=2)
    return {
        "N_images": N,
        "captions": len(all_caps),
        "Distinct-1": float(d1),
        "Distinct-2": float(d2),
        "NCR_mean": float(np.mean(ncrs)),
        "NCR_std": float(np.std(ncrs)),
        "alpha": alpha,
        "tau": tau,
        "k": k,
        "topN": topN
    }


In [21]:
ALPHA = 0.8
TAU   = 0.0
K     = 25.0
TOPN  = 25

metrics_200 = eval_c3dc_on_val(N=200, alpha=ALPHA, tau=TAU, k=K, topN=TOPN)
print(metrics_200)


loading annotations into memory...
Done (t=0.07s)
creating index...
index created!


c3dc eval 200: 100%|█████████████████████████████████████████████████████████████████| 200/200 [02:40<00:00,  1.24it/s]

{'N_images': 200, 'captions': 600, 'Distinct-1': 0.23506382978723403, 'Distinct-2': 0.6145971563981043, 'NCR_mean': 0.9013759514980492, 'NCR_std': 0.1096170498458102, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 25}


In [22]:
from pathlib import Path

# Load your labels.txt as the "noun vocabulary"
LABELS_TXT = Path(r"D:\PROJECT\labels.txt")
LABEL_VOCAB = set([l.strip() for l in LABELS_TXT.read_text(encoding="utf-8").splitlines() if l.strip()])

STOPWORDS = set("""
a an the and or of in on at to for from with by is are was were be been being
this that these those it its as into over under up down out off if then than
not no very so too such just about across after again against all am among
any because before between both but can could did do does doing don during each
few had has have having he her hers him his how i im ive me more most my our ours
she should some their theirs them themselves there they through time until we
what when where which while who whom why will would you your yours yourself
""".split())

def extract_noun_tokens(caption: str):
    toks = []
    for t in caption.lower().split():
        t = "".join([ch for ch in t if ch.isalpha()])
        if len(t) < 3: 
            continue
        if t in STOPWORDS:
            continue
        # keep only tokens that exist in labels vocabulary (proxy for nouns/objects/scenes)
        if t in LABEL_VOCAB:
            toks.append(t)
    return toks

def ncr_noun_only(caption: str, allowed_set: set):
    nouns = extract_noun_tokens(caption)
    if not nouns:
        return 0.0
    bad = sum(1 for w in nouns if w not in allowed_set)
    return bad / len(nouns)


In [23]:
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image

def eval_c3dc_on_val_nounNCR(N=200, alpha=0.8, tau=0.0, k=25.0, topN=25):
    coco_val = COCO(str(VAL_ANN))
    img_ids = list(coco_val.imgs.keys())[:N]

    all_caps = []
    ncrs = []

    for img_id in tqdm(img_ids, desc=f"c3dc eval nounNCR {N}"):
        info = coco_val.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        allowed_set, conf = clip_ctx.get_context(pil, topN=topN)

        for div, cfg in [
            (DIV0, dict(temperature=1.0, top_k=0,  top_p=1.0, rep_penalty=1.15)),
            (DIV1, dict(temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)),
            (DIV2, dict(temperature=1.1, top_k=0,  top_p=0.9, rep_penalty=1.25)),
        ]:
            cap, _ = sample_decode_c3dc(
                model, img_t, stoi, itos, div,
                allowed_set, conf,
                alpha=alpha, tau=tau, k=k,
                **cfg
            )
            all_caps.append(cap)
            ncrs.append(ncr_noun_only(cap, allowed_set))

    d1 = distinct_n(all_caps, n=1)
    d2 = distinct_n(all_caps, n=2)

    return {
        "N_images": N,
        "captions": len(all_caps),
        "Distinct-1": float(d1),
        "Distinct-2": float(d2),
        "NounNCR_mean": float(np.mean(ncrs)),
        "NounNCR_std": float(np.std(ncrs)),
        "alpha": alpha, "tau": tau, "k": k, "topN": topN
    }

ALPHA = 0.8
TAU   = 0.0
K     = 25.0
TOPN  = 25

metrics_200_noun = eval_c3dc_on_val_nounNCR(N=200, alpha=ALPHA, tau=TAU, k=K, topN=TOPN)
print(metrics_200_noun)


loading annotations into memory...
Done (t=0.76s)
creating index...
index created!


c3dc eval nounNCR 200: 100%|█████████████████████████████████████████████████████████| 200/200 [02:48<00:00,  1.19it/s]

{'N_images': 200, 'captions': 600, 'Distinct-1': 0.24284741144414168, 'Distinct-2': 0.6151365705614568, 'NounNCR_mean': 0.7828988095238095, 'NounNCR_std': 0.2534781996856769, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 25}


In [25]:
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image
import numpy as np

@torch.no_grad()
def sample_decode_baseline(model, image_tensor, stoi, itos, div_token,
                           max_len=30, temperature=1.0, top_k=0, top_p=1.0,
                           rep_penalty=1.0, block_bigrams=True):
    # Uses your existing sampler WITHOUT pseudo-context penalty
    cap = sample_decode(model, image_tensor, stoi, itos, div_token,
                        max_len=max_len, temperature=temperature, top_k=top_k, top_p=top_p,
                        rep_penalty=rep_penalty, block_bigrams=block_bigrams)
    return cap

def eval_baseline_on_val_nounNCR(N=200):
    coco_val = COCO(str(VAL_ANN))
    img_ids = list(coco_val.imgs.keys())[:N]

    all_caps = []
    ncrs = []

    for img_id in tqdm(img_ids, desc=f"baseline eval nounNCR {N}"):
        info = coco_val.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        # still compute allowed_set for NCR measurement (but DO NOT use it in decoding)
        allowed_set, conf = clip_ctx.get_context(pil, topN=TOPN)

        for div, cfg in [
            (DIV0, dict(temperature=1.0, top_k=0,  top_p=1.0, rep_penalty=1.15)),
            (DIV1, dict(temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)),
            (DIV2, dict(temperature=1.1, top_k=0,  top_p=0.9, rep_penalty=1.25)),
        ]:
            cap = sample_decode_baseline(model, img_t, stoi, itos, div, **cfg)
            all_caps.append(cap)
            ncrs.append(ncr_noun_only(cap, allowed_set))

    d1 = distinct_n(all_caps, n=1)
    d2 = distinct_n(all_caps, n=2)

    return {
        "N_images": N,
        "captions": len(all_caps),
        "Distinct-1": float(d1),
        "Distinct-2": float(d2),
        "NounNCR_mean": float(np.mean(ncrs)),
        "NounNCR_std": float(np.std(ncrs)),
    }

base_200_noun = eval_baseline_on_val_nounNCR(N=200)
print(base_200_noun)


loading annotations into memory...
Done (t=0.06s)
creating index...
index created!


baseline eval nounNCR 200:   0%|                                                               | 0/200 [00:00<?, ?it/s]


NameError: name 'sample_decode' is not defined

In [26]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def sample_decode(model, image_tensor, stoi, itos, div_token,
                  max_len=30, temperature=1.0, top_k=0, top_p=1.0,
                  rep_penalty=1.0, block_bigrams=True):
    model.eval()
    img = image_tensor.unsqueeze(0).to(DEVICE)
    mem = model.enc(img)

    pad_id = stoi["<PAD>"]
    bos_id = stoi["<BOS>"]
    eos_id = stoi["<EOS>"]
    div_id = stoi[div_token]

    seq = [div_id, bos_id]
    used_bigrams = set()

    for _ in range(max_len):
        inp = torch.tensor(seq, device=DEVICE).unsqueeze(0)
        logits = model.dec(inp, mem, tgt_key_padding_mask=inp.eq(pad_id))[:, -1, :].squeeze(0)

        # repetition penalty
        if rep_penalty and rep_penalty > 1.0:
            for prev in set(seq):
                logits[prev] /= rep_penalty

        # bigram blocking (light)
        if block_bigrams and len(seq) >= 2:
            prev_tok = seq[-1]
            # cheap block for already used bigrams over full vocab
            # (works fine for baseline)
            for cand in range(logits.numel()):
                if (prev_tok, cand) in used_bigrams:
                    logits[cand] = -1e9

        # temperature
        logits = logits / max(1e-6, temperature)
        probs = F.softmax(logits, dim=-1)

        # top-k
        if top_k and top_k > 0:
            v, idx = torch.topk(probs, top_k)
            probs2 = torch.zeros_like(probs)
            probs2[idx] = v
            probs = probs2 / probs2.sum()

        # top-p
        if top_p < 1.0:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            csum = torch.cumsum(sorted_probs, dim=0)
            cut = (csum > top_p).nonzero(as_tuple=False)
            if cut.numel() > 0:
                last = cut[0].item()
                keep = sorted_idx[: last+1]
                probs2 = torch.zeros_like(probs)
                probs2[keep] = probs[keep]
                probs = probs2 / probs2.sum()

        next_id = int(torch.multinomial(probs, 1).item())
        seq.append(next_id)
        used_bigrams.add((seq[-2], seq[-1]))

        if next_id == eos_id:
            break

    # decode tokens
    words = []
    for i in seq:
        tok = itos[i]
        if tok == "<EOS>":
            break
        if tok in ["<PAD>", "<BOS>", "<EOS>", "<UNK>", "<DIV0>", "<DIV1>", "<DIV2>"]:
            continue
        words.append(tok)

    return " ".join(words)


In [27]:
import torch
from pathlib import Path

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")
ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Reloaded epoch:", ckpt.get("epoch"))


Reloaded epoch: 12


In [28]:
base_200_noun = eval_baseline_on_val_nounNCR(N=200)
print("BASE:", base_200_noun)

metrics_200_noun = eval_c3dc_on_val_nounNCR(N=200, alpha=0.8, tau=0.0, k=25.0, topN=25)
print("C3DC:", metrics_200_noun)


loading annotations into memory...
Done (t=0.16s)
creating index...
index created!


baseline eval nounNCR 200: 100%|█████████████████████████████████████████████████████| 200/200 [02:24<00:00,  1.38it/s]


BASE: {'N_images': 200, 'captions': 600, 'Distinct-1': 0.21065949707417364, 'Distinct-2': 0.6092958238686004, 'NounNCR_mean': 0.7079667508417508, 'NounNCR_std': 0.23108413659684934}
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!


c3dc eval nounNCR 200: 100%|█████████████████████████████████████████████████████████| 200/200 [02:44<00:00,  1.22it/s]

C3DC: {'N_images': 200, 'captions': 600, 'Distinct-1': 0.20481724776823312, 'Distinct-2': 0.6093310848791456, 'NounNCR_mean': 0.6497222222222223, 'NounNCR_std': 0.2551277012163547, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 25}


In [29]:
# ============================
# N = 500 evaluation (Baseline vs C3DC)
# Uses your existing functions:
#   - eval_baseline_on_val_nounNCR
#   - eval_c3dc_on_val_nounNCR
# And global params: ALPHA, TAU, K, TOPN
# ============================

N = 500
ALPHA = 0.8
TAU   = 0.0
K     = 25.0
TOPN  = 25

# (Optional but recommended) reload latest checkpoint_last before eval
import torch
from pathlib import Path
RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")
ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Reloaded epoch:", ckpt.get("epoch"))

# ---- Baseline (no constraint) ----
base_500 = eval_baseline_on_val_nounNCR(N=N)
print("\nBASE:", base_500)

# ---- C3DC (with constraint) ----
c3dc_500 = eval_c3dc_on_val_nounNCR(N=N, alpha=ALPHA, tau=TAU, k=K, topN=TOPN)
print("\nC3DC:", c3dc_500)

# ---- Simple deltas ----
delta_ncr = base_500["NounNCR_mean"] - c3dc_500["NounNCR_mean"]
rel_drop = (delta_ncr / base_500["NounNCR_mean"]) * 100.0 if base_500["NounNCR_mean"] > 0 else 0.0

print("\nΔ NounNCR (abs):", delta_ncr)
print("NounNCR reduction (%):", rel_drop)
print("Δ Distinct-1:", base_500["Distinct-1"] - c3dc_500["Distinct-1"])
print("Δ Distinct-2:", base_500["Distinct-2"] - c3dc_500["Distinct-2"])


Reloaded epoch: 12
loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


baseline eval nounNCR 500: 100%|█████████████████████████████████████████████████████| 500/500 [05:50<00:00,  1.43it/s]



BASE: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.15082472241833, 'Distinct-2': 0.5159434699240111, 'NounNCR_mean': 0.6970764550264551, 'NounNCR_std': 0.24407375190219222}
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:54<00:00,  1.21it/s]


C3DC: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1454399357171555, 'Distinct-2': 0.5133606252326014, 'NounNCR_mean': 0.6482801587301588, 'NounNCR_std': 0.24834062625434467, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 25}

Δ NounNCR (abs): 0.04879629629629634
NounNCR reduction (%): 7.000135486493292
Δ Distinct-1: 0.00538478670117451
Δ Distinct-2: 0.0025828446914096803


In [30]:
# ============================
# ABLATION: effect of topN (pseudo-context size)
# Keep alpha/tau/k fixed; compare TOPN=25 vs TOPN=40 at N=500
# ============================

import torch
from pathlib import Path

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")

# reload latest checkpoint
ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Reloaded epoch:", ckpt.get("epoch"))

N = 500
ALPHA = 0.8
TAU   = 0.0
K     = 25.0

def run_pair(topN):
    global TOPN
    TOPN = topN  # used by eval_baseline_on_val_nounNCR if it references TOPN
    print(f"\n===== topN={topN} =====")

    base = eval_baseline_on_val_nounNCR(N=N)
    print("BASE:", base)

    c3dc = eval_c3dc_on_val_nounNCR(N=N, alpha=ALPHA, tau=TAU, k=K, topN=topN)
    print("C3DC:", c3dc)

    delta_ncr = base["NounNCR_mean"] - c3dc["NounNCR_mean"]
    rel_drop = (delta_ncr / base["NounNCR_mean"]) * 100.0 if base["NounNCR_mean"] > 0 else 0.0

    print("Δ NounNCR (abs):", delta_ncr)
    print("NounNCR reduction (%):", rel_drop)
    print("Δ Distinct-1:", base["Distinct-1"] - c3dc["Distinct-1"])
    print("Δ Distinct-2:", base["Distinct-2"] - c3dc["Distinct-2"])

    return base, c3dc

# Run ablation
base25, c3dc25 = run_pair(25)
base40, c3dc40 = run_pair(40)


Reloaded epoch: 12

===== topN=25 =====
loading annotations into memory...
Done (t=0.47s)
creating index...
index created!


baseline eval nounNCR 500: 100%|█████████████████████████████████████████████████████| 500/500 [05:47<00:00,  1.44it/s]


BASE: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.14982354828360603, 'Distinct-2': 0.5206247781327653, 'NounNCR_mean': 0.6968282587782587, 'NounNCR_std': 0.23624957873044714}
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:55<00:00,  1.20it/s]


C3DC: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1463887218547471, 'Distinct-2': 0.5142199450508651, 'NounNCR_mean': 0.6445585470085471, 'NounNCR_std': 0.25262443740801244, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 25}
Δ NounNCR (abs): 0.052269711769711624
NounNCR reduction (%): 7.501089559908998
Δ Distinct-1: 0.0034348264288589225
Δ Distinct-2: 0.0064048330819002075

===== topN=40 =====
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


baseline eval nounNCR 500: 100%|█████████████████████████████████████████████████████| 500/500 [05:57<00:00,  1.40it/s]


BASE: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1472276540717845, 'Distinct-2': 0.5130634710513482, 'NounNCR_mean': 0.6464044973544973, 'NounNCR_std': 0.24515683476723016}
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:56<00:00,  1.20it/s]

C3DC: {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1463055834169174, 'Distinct-2': 0.5123745819397993, 'NounNCR_mean': 0.5840841269841269, 'NounNCR_std': 0.2476329179960731, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 40}
Δ NounNCR (abs): 0.062320370370370415
NounNCR reduction (%): 9.64107932810267
Δ Distinct-1: 0.000922070654867102
Δ Distinct-2: 0.0006888891115488516


In [57]:
# ============================
# FINAL ROBUSTNESS EVAL
# N = 2000, topN = 40
# ============================

import torch
from pathlib import Path

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")

# reload latest checkpoint
ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Reloaded epoch:", ckpt.get("epoch"))

# settings
N = 2000
ALPHA = 0.8
TAU   = 0.0
K     = 25.0
TOPN  = 40

# ---- Baseline ----
base_2000 = eval_baseline_on_val_nounNCR(N=N)
print("\nBASE:", base_2000)

# ---- C3DC ----
c3dc_2000 = eval_c3dc_on_val_nounNCR(
    N=N, alpha=ALPHA, tau=TAU, k=K, topN=TOPN
)
print("\nC3DC:", c3dc_2000)

# ---- Deltas ----
delta_ncr = base_2000["NounNCR_mean"] - c3dc_2000["NounNCR_mean"]
rel_drop = (delta_ncr / base_2000["NounNCR_mean"]) * 100.0

print("\nΔ NounNCR (abs):", delta_ncr)
print("NounNCR reduction (%):", rel_drop)
print("Δ Distinct-1:", base_2000["Distinct-1"] - c3dc_2000["Distinct-1"])
print("Δ Distinct-2:", base_2000["Distinct-2"] - c3dc_2000["Distinct-2"])


Reloaded epoch: 18
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!


baseline eval nounNCR 2000: 100%|██████████████████████████████████████████████████| 2000/2000 [22:35<00:00,  1.48it/s]



BASE: {'N_images': 2000, 'captions': 6000, 'Distinct-1': 0.08049705478899379, 'Distinct-2': 0.38291789511301705, 'NounNCR_mean': 0.6369876142376143, 'NounNCR_std': 0.24779666167213968}
loading annotations into memory...
Done (t=0.64s)
creating index...
index created!


c3dc eval nounNCR 2000: 100%|██████████████████████████████████████████████████████| 2000/2000 [24:56<00:00,  1.34it/s]


C3DC: {'N_images': 2000, 'captions': 6000, 'Distinct-1': 0.07571633843293044, 'Distinct-2': 0.3718488975844684, 'NounNCR_mean': 0.5765896584896585, 'NounNCR_std': 0.25780348544284243, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 40}

Δ NounNCR (abs): 0.06039795574795581
NounNCR reduction (%): 9.481810069453827
Δ Distinct-1: 0.00478071635606335
Δ Distinct-2: 0.011068997528548674


In [58]:
# ============================
# Load epoch 13 checkpoint
# ============================
import torch
from pathlib import Path

RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")

ckpt = torch.load(RUN_DIR / "checkpoint_last.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

print("Loaded epoch:", ckpt.get("epoch"))


Loaded epoch: 18


In [34]:
!pip install pycocoevalcap


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
from pycocoevalcap.eval import COCOEvalCap
import json
import tempfile

def eval_standard_metrics(N=500, use_c3dc=False,
                          alpha=0.8, tau=0.0, k=25.0, topN=40):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    results = []
    gts = {}

    for img_id in img_ids:
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        caps = []
        if use_c3dc:
            allowed_set, conf = clip_ctx.get_context(pil, topN=topN)
            for div in [DIV0, DIV1, DIV2]:
                cap, _ = sample_decode_c3dc(
                    model, img_t, stoi, itos, div,
                    allowed_set, conf,
                    temperature=1.0, top_k=30, top_p=0.9,
                    rep_penalty=1.2
                )
                caps.append(cap)
        else:
            for div in [DIV0, DIV1, DIV2]:
                caps.append(sample_decode(
                    model, img_t, stoi, itos, div
                ))

        results.append({
            "image_id": img_id,
            "caption": caps[0]   # COCOEvalCap expects 1 caption/image
        })
        gts[img_id] = coco.imgToAnns[img_id]

    # temporary files
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(results, f)
        res_file = f.name

    cocoRes = coco.loadRes(res_file)
    cocoEval = COCOEvalCap(coco, cocoRes)
    cocoEval.evaluate()

    return {
        "CIDEr": cocoEval.eval["CIDEr"],
        "BLEU-4": cocoEval.eval["Bleu_4"]
    }



In [37]:
import json, tempfile
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from PIL import Image
from tqdm import tqdm

def eval_standard_metrics_subset(
    N=500,
    use_c3dc=False,
    alpha=0.8, tau=0.0, k=25.0, topN=40,
    use_div_caption="DIV0"  # choose which caption to score per image
):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    div_map = {"DIV0": DIV0, "DIV1": DIV1, "DIV2": DIV2}
    div_token = div_map[use_div_caption]

    results = []

    for img_id in tqdm(img_ids, desc=f"gen caps N={N} use_c3dc={use_c3dc}"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        if use_c3dc:
            allowed_set, conf = clip_ctx.get_context(pil, topN=topN)
            cap, _ = sample_decode_c3dc(
                model, img_t, stoi, itos, div_token,
                allowed_set, conf,
                alpha=alpha, tau=tau, k=k,
                # decoding defaults (keep stable for eval)
                temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2
            )
        else:
            cap = sample_decode(
                model, img_t, stoi, itos, div_token,
                max_len=30, temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2
            )

        results.append({"image_id": int(img_id), "caption": cap})

    # write results json
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, encoding="utf-8") as f:
        json.dump(results, f)
        res_file = f.name

    cocoRes = coco.loadRes(res_file)
    cocoEval = COCOEvalCap(coco, cocoRes)

    # IMPORTANT: restrict eval to the same subset
    cocoEval.params["image_id"] = img_ids

    cocoEval.evaluate()

    return {
        "N": N,
        "caption_used": use_div_caption,
        "BLEU-4": float(cocoEval.eval["Bleu_4"]),
        "CIDEr": float(cocoEval.eval["CIDEr"]),
        "METEOR": float(cocoEval.eval.get("METEOR", 0.0)),
        "ROUGE_L": float(cocoEval.eval.get("ROUGE_L", 0.0)),
    }


In [44]:
from pycocotools.coco import COCO
from PIL import Image
from tqdm import tqdm

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

def eval_standard_metrics_robust(
    N=500,
    use_c3dc=False,
    alpha=0.8, tau=0.0, k=25.0, topN=40,
    use_div_caption="DIV0",
    include_meteor=False  # keep False on Windows
):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    div_map = {"DIV0": DIV0, "DIV1": DIV1, "DIV2": DIV2}
    div_token = div_map[use_div_caption]

    gts, res = {}, {}

    for img_id in tqdm(img_ids, desc=f"std-metrics N={N} c3dc={use_c3dc}"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        if use_c3dc:
            allowed_set, conf = clip_ctx.get_context(pil, topN=topN)
            hyp, _ = sample_decode_c3dc(
                model, img_t, stoi, itos, div_token,
                allowed_set, conf,
                alpha=alpha, tau=tau, k=k,
                temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2
            )
        else:
            hyp = sample_decode(
                model, img_t, stoi, itos, div_token,
                max_len=30, temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2
            )

        refs = [a["caption"] for a in coco.imgToAnns[img_id]]
        gts[img_id] = refs
        res[img_id] = [hyp]

    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr"),
    ]

    out = {"N": N, "caption_used": use_div_caption}

    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for s, m in zip(score, method):
                out[m] = float(s)
        else:
            out[method] = float(score)

    # Optional METEOR (disabled by default because it's flaky on many Windows setups)
    if include_meteor:
        try:
            from pycocoevalcap.meteor.meteor import Meteor
            m = Meteor()
            score, _ = m.compute_score(gts, res)
            out["METEOR"] = float(score)
        except Exception as e:
            out["METEOR"] = None
            out["METEOR_error"] = str(e)

    return out


In [48]:
base_std = eval_standard_metrics_robust(N=500, use_c3dc=False, use_div_caption="DIV0", topN=40)
c3dc_std = eval_standard_metrics_robust(N=500, use_c3dc=True,  use_div_caption="DIV0",
                                       alpha=0.8, tau=0.0, k=25.0, topN=40)

print("BASE_STD:", base_std)
print("C3DC_STD:", c3dc_std)


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


std-metrics N=500 c3dc=False: 100%|██████████████████████████████████████████████████| 500/500 [01:43<00:00,  4.85it/s]


{'testlen': 5043, 'reflen': 4929, 'guess': [5043, 4543, 4043, 3543], 'correct': [2756, 861, 264, 90]}
ratio: 1.0231284236151301
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!


std-metrics N=500 c3dc=True: 100%|███████████████████████████████████████████████████| 500/500 [02:12<00:00,  3.77it/s]


{'testlen': 4811, 'reflen': 4822, 'guess': [4811, 4311, 3811, 3311], 'correct': [2713, 921, 301, 96]}
ratio: 0.9977187888840734
BASE_STD: {'N': 500, 'caption_used': 'DIV0', 'Bleu_1': 0.5465000991472245, 'Bleu_2': 0.3218291141723479, 'Bleu_3': 0.18911106848453488, 'Bleu_4': 0.11448683530773066, 'ROUGE_L': 0.3460476260940928, 'CIDEr': 0.46964543312234414}
C3DC_STD: {'N': 500, 'caption_used': 'DIV0', 'Bleu_1': 0.5626281458645014, 'Bleu_2': 0.346302031702657, 'Bleu_3': 0.21142104466061265, 'Bleu_4': 0.12858518327943777, 'ROUGE_L': 0.35054507315508493, 'CIDEr': 0.4956612973613678}


In [49]:
COCO_OBJECTS = set([
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat",
    "bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe",
    "backpack","umbrella","handbag","tie","suitcase",
    "frisbee","skis","snowboard","kite","skateboard","surfboard","tennis",
    "bottle","cup","fork","knife","spoon","bowl",
    "banana","apple","sandwich","orange","broccoli","carrot","pizza","donut","cake",
    "chair","couch","bed","table","toilet","tv","laptop","mouse","keyboard","phone",
    "microwave","oven","sink","refrigerator","book","clock","vase","scissors","teddy"
])

def extract_obj_words(caption):
    toks = [t.lower().strip(".,!?;:()[]{}'\"") for t in caption.split()]
    return set([t for t in toks if t in COCO_OBJECTS])

def obj_precision_vs_context(caption, allowed_set):
    objs = extract_obj_words(caption)
    if not objs:
        return 1.0  # no object words => not hallucinating objects
    return len(objs & allowed_set) / len(objs)


In [60]:
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image

def eval_obj_precision(N=500, use_c3dc=False, alpha=0.8, tau=0.0, k=25.0, topN=40):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]
    scores = []

    for img_id in tqdm(img_ids, desc=f"obj-prec N={N} c3dc={use_c3dc}"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        allowed_set, conf = clip_ctx.get_context(pil, topN=topN)

        # evaluate on all 3 captions like your diversity evaluation
        for div, cfg in [
            (DIV0, dict(temperature=1.0, top_k=0,  top_p=1.0, rep_penalty=1.15)),
            (DIV1, dict(temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)),
            (DIV2, dict(temperature=1.1, top_k=0,  top_p=0.9, rep_penalty=1.25)),
        ]:
            if use_c3dc:
                cap, _ = sample_decode_c3dc(model, img_t, stoi, itos, div,
                                            allowed_set, conf,
                                            alpha=alpha, tau=tau, k=k, **cfg)
            else:
                cap = sample_decode(model, img_t, stoi, itos, div, **cfg)

            scores.append(obj_precision_vs_context(cap, allowed_set))

    return {"N": N, "captions": len(scores),
            "ObjPrec_mean": float(np.mean(scores)),
            "ObjPrec_std": float(np.std(scores))}

obj_base = eval_obj_precision(N=500, use_c3dc=False, topN=40)
obj_c3dc = eval_obj_precision(N=500, use_c3dc=True, alpha=0.8, tau=0.0, k=25.0, topN=40)

print("OBJ_BASE:", obj_base)
print("OBJ_C3DC:", obj_c3dc)


loading annotations into memory...
Done (t=0.48s)
creating index...
index created!


obj-prec N=500 c3dc=False: 100%|█████████████████████████████████████████████████████| 500/500 [05:50<00:00,  1.43it/s]


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


obj-prec N=500 c3dc=True: 100%|██████████████████████████████████████████████████████| 500/500 [06:56<00:00,  1.20it/s]

OBJ_BASE: {'N': 500, 'captions': 1500, 'ObjPrec_mean': 0.8190555555555556, 'ObjPrec_std': 0.359893035377986}
OBJ_C3DC: {'N': 500, 'captions': 1500, 'ObjPrec_mean': 0.842, 'ObjPrec_std': 0.3360429866682072}


In [56]:
import random
from pycocotools.coco import COCO
from PIL import Image

def qualitative_dump(Kimgs=8, seed=7, topN=40, alpha=0.8, tau=0.0, k=25.0):
    random.seed(seed)
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())
    picks = random.sample(img_ids, Kimgs)

    for j, img_id in enumerate(picks, 1):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        allowed_set, conf = clip_ctx.get_context(pil, topN=topN)
        lam = float(1 / (1 + np.exp(-(k*(conf - tau)))))  # sigmoid

        top_ctx = sorted(list(allowed_set))[:10]

        print(f"\n=== Example {j} | image_id={img_id} | file={info['file_name']} ===")
        print("pseudo-context(top10):", top_ctx)
        print("conf:", float(conf), "lambda(sigmoid):", lam)

        for div in [DIV0, DIV1, DIV2]:
            base = sample_decode(model, img_t, stoi, itos, div,
                                 max_len=30, temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2)
            c3, _ = sample_decode_c3dc(model, img_t, stoi, itos, div,
                                       allowed_set, conf,
                                       alpha=alpha, tau=tau, k=k,
                                       temperature=1.0, top_k=30, top_p=0.9, rep_penalty=1.2)
            print(f"\n{div}:")
            print("  BASE :", base)
            print("  C3DC :", c3)

qualitative_dump(Kimgs=8, topN=40, alpha=0.8, tau=0.0, k=25.0)


loading annotations into memory...
Done (t=2.01s)
creating index...
index created!

=== Example 1 | image_id=125778 | file=000000125778.jpg ===
pseudo-context(top10): ['apartment', 'area', 'bedroom', 'chair', 'chairs', 'comforter', 'corner', 'couch', 'couches', 'curtains']
conf: 0.00852513313293457 lambda(sigmoid): 0.5530813049952864

<DIV0>:
  BASE : a living room filled with furniture and a fire place
  C3DC : a living room with couch and chair in it

<DIV1>:
  BASE : a woman is sitting in the living room with her laptop
  C3DC : a man sitting in the living room looking at his laptop

<DIV2>:
  BASE : a man looking at a laptop computer in an otherwise room
  C3DC : a living room with couch chair and ottoman

=== Example 2 | image_id=92177 | file=000000092177.jpg ===
pseudo-context(top10): ['baked', 'bakery', 'baking', 'birthday', 'blue', 'bow', 'cake', 'cakes', 'chef', 'cupcake']
conf: 0.020815759897232056 lambda(sigmoid): 0.6272398914736826

<DIV0>:
  BASE : a cake made in black and

In [52]:
for a in [0.6, 0.8, 1.0]:
    res = eval_c3dc_on_val_nounNCR(N=500, alpha=a, tau=0.0, k=25.0, topN=40)
    print("\nalpha=", a, res)


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:16<00:00,  1.33it/s]



alpha= 0.6 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.14644632540642213, 'Distinct-2': 0.5047811145973405, 'NounNCR_mean': 0.5890031746031745, 'NounNCR_std': 0.256923618813238, 'alpha': 0.6, 'tau': 0.0, 'k': 25.0, 'topN': 40}
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:12<00:00,  1.34it/s]



alpha= 0.8 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1495891899232702, 'Distinct-2': 0.516443638013155, 'NounNCR_mean': 0.5679334147334147, 'NounNCR_std': 0.26392979514772585, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 40}
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:02<00:00,  1.38it/s]


alpha= 1.0 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.14268900581556357, 'Distinct-2': 0.501004326328801, 'NounNCR_mean': 0.5528293650793651, 'NounNCR_std': 0.2619740127401571, 'alpha': 1.0, 'tau': 0.0, 'k': 25.0, 'topN': 40}


In [53]:
for kk in [10.0, 25.0, 50.0]:
    res = eval_c3dc_on_val_nounNCR(N=500, alpha=0.8, tau=0.0, k=kk, topN=40)
    print("\nk=", kk, res)


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [05:57<00:00,  1.40it/s]



k= 10.0 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1482389808570066, 'Distinct-2': 0.511874952576068, 'NounNCR_mean': 0.5829201058201059, 'NounNCR_std': 0.2552734084123688, 'alpha': 0.8, 'tau': 0.0, 'k': 10.0, 'topN': 40}
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|███████████████████████████████████████████████| 500/500 [06:09<00:00,  1.35it/s]



k= 25.0 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.14536256323777402, 'Distinct-2': 0.5055909943714821, 'NounNCR_mean': 0.5732120731120731, 'NounNCR_std': 0.2634120382488842, 'alpha': 0.8, 'tau': 0.0, 'k': 25.0, 'topN': 40}
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


c3dc eval nounNCR 500: 100%|█████████████████████████████████████████████████████████| 500/500 [06:10<00:00,  1.35it/s]


k= 50.0 {'N_images': 500, 'captions': 1500, 'Distinct-1': 0.1508881780439665, 'Distinct-2': 0.5194785508564499, 'NounNCR_mean': 0.5604830687830687, 'NounNCR_std': 0.2633084364355475, 'alpha': 0.8, 'tau': 0.0, 'k': 50.0, 'topN': 40}


In [54]:
def sample_decode_c3dc_fixed_lambda(model, image_tensor, stoi, itos, div_token,
                                    allowed_set, fixed_lam=0.5,
                                    alpha=0.8,
                                    **kwargs):
    # Call your existing sample_decode_c3dc but pretend conf produces constant lambda.
    # We do this by passing a fake conf that makes sigmoid(...) ~ fixed_lam.
    # Invert sigmoid: x = log(lam/(1-lam)) => conf = x/k + tau
    # We'll use k=25, tau=0 here.
    import math
    lam = float(fixed_lam)
    lam = min(max(lam, 1e-4), 1-1e-4)
    x = math.log(lam/(1-lam))
    fake_conf = x / 25.0  # tau=0, k=25

    return sample_decode_c3dc(model, image_tensor, stoi, itos, div_token,
                              allowed_set, fake_conf,
                              alpha=alpha, tau=0.0, k=25.0,
                              **kwargs)


In [55]:
import numpy as np
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image

def eval_fixed_vs_calibrated(N=500, topN=40, alpha=0.8, fixed_lam=0.5):
    coco = COCO(str(VAL_ANN))
    img_ids = list(coco.imgs.keys())[:N]

    ncr_fixed, ncr_cal = [], []

    for img_id in tqdm(img_ids, desc=f"fixed-vs-cal N={N}"):
        info = coco.loadImgs([img_id])[0]
        pil = Image.open(VAL_IMG / info["file_name"]).convert("RGB")
        img_t = tfm(pil)

        allowed_set, conf = clip_ctx.get_context(pil, topN=topN)

        for div, cfg in [
            (DIV0, dict(temperature=1.0, top_k=0,  top_p=1.0, rep_penalty=1.15)),
            (DIV1, dict(temperature=0.9, top_k=30, top_p=1.0, rep_penalty=1.2)),
            (DIV2, dict(temperature=1.1, top_k=0,  top_p=0.9, rep_penalty=1.25)),
        ]:
            cap_fixed, _ = sample_decode_c3dc_fixed_lambda(
                model, img_t, stoi, itos, div, allowed_set,
                fixed_lam=fixed_lam, alpha=alpha, **cfg
            )
            cap_cal, _ = sample_decode_c3dc(
                model, img_t, stoi, itos, div, allowed_set, conf,
                alpha=alpha, tau=0.0, k=25.0, **cfg
            )

            ncr_fixed.append(ncr_noun_only(cap_fixed, allowed_set))
            ncr_cal.append(ncr_noun_only(cap_cal, allowed_set))

    return {
        "N": N,
        "topN": topN,
        "alpha": alpha,
        "fixed_lam": fixed_lam,
        "Fixed_mean": float(np.mean(ncr_fixed)),
        "Calibrated_mean": float(np.mean(ncr_cal)),
        "Improvement_abs": float(np.mean(ncr_fixed) - np.mean(ncr_cal)),
    }

fixed_vs = eval_fixed_vs_calibrated(N=500, topN=40, alpha=0.8, fixed_lam=0.5)
print(fixed_vs)


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


fixed-vs-cal N=500: 100%|████████████████████████████████████████████████████████████| 500/500 [12:24<00:00,  1.49s/it]

{'N': 500, 'topN': 40, 'alpha': 0.8, 'fixed_lam': 0.5, 'Fixed_mean': 0.5970528379028378, 'Calibrated_mean': 0.5677400913900914, 'Improvement_abs': 0.0293127465127464}


In [2]:
from pathlib import Path
import torch
import json
import re
from collections import defaultdict, Counter
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------
# Adjust these paths if needed
# -------------------------------------------------
RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")  # <-- CHANGE
CKPT_PATH = RUN_DIR / "checkpoint_best.pt"
COCO_INSTANCES = Path(r"C:\COCO\annotations\instances_val2017.json")

assert CKPT_PATH.exists(), "checkpoint_best.pt not found!"
assert COCO_INSTANCES.exists(), "instances_val2017.json not found!"

print("Using:", CKPT_PATH)


Using: D:\PROJECT\runs\c3dc_baseline_20260210_084907\checkpoint_best.pt


In [3]:
with open(COCO_INSTANCES, "r") as f:
    inst = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in inst["categories"]}

imgid_to_gtcats = defaultdict(set)
for ann in inst["annotations"]:
    imgid = ann["image_id"]
    cname = cat_id_to_name[ann["category_id"]]
    imgid_to_gtcats[imgid].add(cname)

print("GT object map built for:", len(imgid_to_gtcats), "images")


GT object map built for: 4952 images


In [4]:
_word_re = re.compile(r"[a-z]+")

def tokenize(text):
    return _word_re.findall(text.lower())

def make_bigrams(tokens):
    return [" ".join(tokens[i:i+2]) for i in range(len(tokens)-1)]

coco_objects = set(cat_id_to_name.values())
coco_unigrams = set([o for o in coco_objects if len(o.split()) == 1])
coco_bigrams = set([o for o in coco_objects if len(o.split()) == 2])

def extract_mentions(caption):
    toks = tokenize(caption)
    mentions = []

    bigrams = make_bigrams(toks)
    used = set()

    for i, bg in enumerate(bigrams):
        if bg in coco_bigrams:
            mentions.append(bg)
            used.add(i)
            used.add(i+1)

    for i, t in enumerate(toks):
        if i not in used and t in coco_unigrams:
            mentions.append(t)

    return mentions


In [5]:
def compute_chair_i(caps_by_imgid):
    total = 0
    hall = 0

    for imgid, cap in caps_by_imgid.items():
        gt_objs = imgid_to_gtcats.get(int(imgid), set())
        mentioned = extract_mentions(cap)

        for obj in mentioned:
            total += 1
            if obj not in gt_objs:
                hall += 1

    chair_i = hall / total if total > 0 else 0.0
    return chair_i, hall, total


In [6]:
# Example: Use same image IDs as NounNCR
IMG_IDS = VAL_IMAGE_IDS_2000  # <-- confirm your variable

caps_base = {}
caps_c4dd = {}

for imgid in tqdm(IMG_IDS):
    caps_base[imgid] = decode_base(imgid)  # <-- your base decode
    caps_c4dd[imgid] = decode_c4dd(imgid)  # <-- your C4DD decode


NameError: name 'VAL_IMAGE_IDS_2000' is not defined

In [7]:
import random

# Load val image ids from instances file (already loaded earlier)
all_val_imgids = list(imgid_to_gtcats.keys())

print("Total val images:", len(all_val_imgids))

# Fix seed for reproducibility
SEED = 42
random.seed(SEED)

N = 2000
VAL_IMAGE_IDS_2000 = random.sample(all_val_imgids, N)

print("Selected", len(VAL_IMAGE_IDS_2000), "random validation images")
print("First 10 IDs:", VAL_IMAGE_IDS_2000[:10])


Total val images: 4952
Selected 2000 random validation images
First 10 IDs: [166391, 114770, 308328, 61418, 345261, 241326, 143931, 351331, 456394, 384136]


In [8]:
import random

SEED = 42
N = 2000
random.seed(SEED)

all_val_imgids = list(imgid_to_gtcats.keys())
VAL_IMAGE_IDS_2000 = random.sample(all_val_imgids, N)

print(f"✅ Random split ready: N={len(VAL_IMAGE_IDS_2000)} seed={SEED}")
print("First 10 ids:", VAL_IMAGE_IDS_2000[:10])


✅ Random split ready: N=2000 seed=42
First 10 ids: [166391, 114770, 308328, 61418, 345261, 241326, 143931, 351331, 456394, 384136]


In [9]:
import re

_word_re = re.compile(r"[a-z]+")

def _tokens(text: str):
    return _word_re.findall(text.lower())

def _bigrams(tokens):
    return [" ".join(tokens[i:i+2]) for i in range(len(tokens)-1)]

# COCO category names from instances file
coco_objects = set(cat_id_to_name.values())
coco_unigrams = set([o for o in coco_objects if len(o.split()) == 1])
coco_bigrams  = set([o for o in coco_objects if len(o.split()) == 2])

# small synonym normalization (optional but helps COCO matching)
_syn = {
    "television": "tv",
    "cellphone": "cell phone",
    "phone": "cell phone",
    "sofa": "couch",
    "bike": "bicycle",
    "bikes": "bicycle",
    "motorbike": "motorcycle",
}

def _norm(tok: str):
    return _syn.get(tok, tok)

def extract_coco_mentions(caption: str):
    toks = [_norm(t) for t in _tokens(caption)]
    mentions = []

    # bigram first
    bigrams = _bigrams(toks)
    used = [False] * len(toks)

    for i, bg in enumerate(bigrams):
        if bg in coco_bigrams:
            mentions.append(bg)
            used[i] = True
            used[i+1] = True

    # then unigram
    for i, t in enumerate(toks):
        if not used[i] and t in coco_unigrams:
            mentions.append(t)

    return mentions

def compute_CHAIR(caps_by_imgid: dict):
    """
    Returns CHAIRi and CHAIRs
      - CHAIRi: hallucinated object mentions / total object mentions
      - CHAIRs: sentences with >=1 hallucinated object / total sentences
    """
    total_mentions = 0
    hall_mentions = 0
    total_sents = 0
    hall_sents = 0

    for imgid, cap in caps_by_imgid.items():
        total_sents += 1
        gt = imgid_to_gtcats.get(int(imgid), set())
        mentioned = extract_coco_mentions(cap)

        sent_has_hall = False
        for obj in mentioned:
            total_mentions += 1
            if obj not in gt:
                hall_mentions += 1
                sent_has_hall = True

        if sent_has_hall:
            hall_sents += 1

    chair_i = hall_mentions / total_mentions if total_mentions > 0 else 0.0
    chair_s = hall_sents / total_sents if total_sents > 0 else 0.0

    return {
        "CHAIRi": chair_i,
        "CHAIRs": chair_s,
        "hall_mentions": hall_mentions,
        "total_mentions": total_mentions,
        "hall_sents": hall_sents,
        "total_sents": total_sents,
    }


In [10]:
from tqdm import tqdm

def generate_caps(img_ids, decode_fn):
    out = {}
    for imgid in tqdm(img_ids):
        out[int(imgid)] = decode_fn(int(imgid))
    return out

# ---- Replace these with your actual decoding functions ----
# BASE decode (no suppression)
caps_base_2000 = generate_caps(VAL_IMAGE_IDS_2000, decode_base)   # <-- REPLACE

# C4DD decode (suppression ON) using your default/best config (alpha=0.8, k=25, tau=0)
caps_c4dd_2000 = generate_caps(VAL_IMAGE_IDS_2000, decode_c4dd)   # <-- REPLACE


NameError: name 'decode_base' is not defined

In [ ]:
res_base = compute_CHAIR(caps_base_2000)
res_c4dd = compute_CHAIR(caps_c4dd_2000)

print("BASE : CHAIRi={:.5f} ({} / {}) | CHAIRs={:.5f} ({} / {})".format(
    res_base["CHAIRi"], res_base["hall_mentions"], res_base["total_mentions"],
    res_base["CHAIRs"], res_base["hall_sents"], res_base["total_sents"]
))
print("C4DD : CHAIRi={:.5f} ({} / {}) | CHAIRs={:.5f} ({} / {})".format(
    res_c4dd["CHAIRi"], res_c4dd["hall_mentions"], res_c4dd["total_mentions"],
    res_c4dd["CHAIRs"], res_c4dd["hall_sents"], res_c4dd["total_sents"]
))


In [11]:
RUN_DIR = Path(r"D:\PROJECT\runs\c3dc_baseline_20260210_084907")  # keep your run
ckpt = torch.load(RUN_DIR / "checkpoint_best.pt", map_location=DEVICE)  # <-- BEST
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Reloaded BEST epoch:", ckpt.get("epoch"))


NameError: name 'DEVICE' is not defined

In [12]:
class CaptionDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.1, max_len=64):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.dec = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt_ids, memory, tgt_key_padding_mask=None):
        B,T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0).expand(B,T)
        x = self.tok_emb(tgt_ids) + self.pos_emb(pos)

        causal = torch.triu(torch.ones(T,T, device=tgt_ids.device), diagonal=1).bool()
        y = self.dec(
            tgt=x,
            memory=memory,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.out(y)


NameError: name 'nn' is not defined

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
